In [0]:
%pip install -U datasets transformers tf-keras accelerate mlflow torchvision deepspeed
dbutils.library.restartPython()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.2 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.7/419.7 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.8/433.8 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.8/220.8 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
from io import BytesIO
from PIL import Image
import pandas as pd
import torch
import mlflow
import mlflow.pyfunc
from pyspark.sql.functions import pandas_udf, col
from datasets import Dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification, Trainer, TrainingArguments, pipeline
from mlflow.models import infer_signature

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a538c5e0-e73b-4ef2-b59e-47d03deab9bc/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


In [0]:
mlflow.set_experiment("/Shared/insurce_claims_experiment")

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2741300458728716', creation_time=1775944404217, experiment_id='2741300458728716', last_update_time=1776976858654, lifecycle_stage='active', name='/Shared/insurce_claims_experiment', tags={'mlflow.experiment.sourceName': '/Shared/insurce_claims_experiment',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'juanpablojpdc2001@gmail.com',
 'mlflow.ownerId': '74841380922543'}, trace_location=None, workspace='default'>

In [0]:
# =====================================================
# Paso 3.5: Definir las clases del modelo
# =====================================================
# Basado en el análisis de training_images, tenemos 3 clases:
# - ok: vehículo sin daño
# - minor: daño menor
# - major: daño mayor

labels = ["ok", "minor", "major"]
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

print("✓ Clases del modelo definidas")
print(f"Número de clases: {len(labels)}")
print(f"Labels: {labels}")
print(f"id2label: {id2label}")
print(f"label2id: {label2id}")

✓ Clases del modelo definidas
Número de clases: 3
Labels: ['ok', 'minor', 'major']
id2label: {0: 'ok', 1: 'minor', 2: 'major'}
label2id: {'ok': 0, 'minor': 1, 'major': 2}


In [0]:
# =====================================================
# Paso 7: Cargar modelo ResNet-50 - ENTRENAR TODO
# =====================================================

checkpoint = "microsoft/resnet-50"

processor = AutoImageProcessor.from_pretrained(checkpoint)

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("✓ Modelo base cargado")

# NO FREEZING - entrenar todas las capas
# Esta es la estrategia del baseline que logró 66.67% accuracy
for param in model.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✓ Modelo configurado para entrenamiento COMPLETO")
print(f"  Total parámetros: {total_params:,}")
print(f"  Parámetros entrenables: {trainable_params:,} (100%)")
print(f"\n⭐ Estrategia: Baseline Extendido - entrenar todo el modelo")

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `1000`.


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

ResNetForImageClassification LOAD REPORT from: microsoft/resnet-50
Key                 | Status   |                                                                                            
--------------------+----------+--------------------------------------------------------------------------------------------
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 2048]) vs model:torch.Size([3, 2048])
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([3])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


✓ Modelo base cargado

✓ Modelo configurado para entrenamiento COMPLETO
  Total parámetros: 23,514,179
  Parámetros entrenables: 23,514,179 (100%)

⭐ Estrategia: Baseline Extendido - entrenar todo el modelo


In [0]:
# =====================================================
# Paso 4: Preparar las imágenes para el entrenamiento
# =====================================================
# Extraemos el label del nombre del archivo y creamos una tabla lista para entrenar

from pyspark.sql.functions import regexp_extract, split, element_at, col

# Leer tabla de training images desde Bronze
df_training_raw = spark.table("smart_claims.bronze.training_images")

# Extraer el nombre del archivo del path
df_training_with_labels = df_training_raw.withColumn(
    "filename",
    element_at(split(col("path"), "/"), -1)
).withColumn(
    "label",
    regexp_extract(col("filename"), r"\d+-(\w+)\s*\(", 1)
)

# Verificar que todas las imágenes tienen label
print("Verificación de datos de entrenamiento:")
total_images = df_training_with_labels.count()
images_with_label = df_training_with_labels.filter(col("label") != "").count()
images_without_label = total_images - images_with_label

print(f"  Total imágenes: {total_images}")
print(f"  Imágenes con label: {images_with_label}")
print(f"  Imágenes sin label: {images_without_label}")

# Distribución por clase
print("\nDistribución por clase:")
display(df_training_with_labels.groupBy("label").count().orderBy("label"))

# Guardar en Silver como tabla de training lista
df_training_with_labels.write.mode("overwrite").saveAsTable("smart_claims.silver.training_images")
print("\n✓ Tabla smart_claims.silver.training_images creada")

Verificación de datos de entrenamiento:
  Total imágenes: 56
  Imágenes con label: 56
  Imágenes sin label: 0

Distribución por clase:


label,count
major,18
minor,21
ok,17



✓ Tabla smart_claims.silver.training_images creada


In [0]:
# =====================================================
# Paso 5-6: Construir dataset y convertir binario a imagen
# =====================================================

from datasets import Dataset, Features, Image as HFImage, ClassLabel
from io import BytesIO
from PIL import Image
import pandas as pd

# Leer la tabla de training desde Silver
df_training = spark.table("smart_claims.silver.training_images").select("content", "label")

# Convertir a pandas (el dataset es pequeño: 56 imágenes)
df_pandas = df_training.toPandas()

print(f"Dataset cargado: {len(df_pandas)} imágenes")
print(f"\nDistribución por clase:")
print(df_pandas['label'].value_counts())

# Función para convertir bytes a imagen PIL
def bytes_to_pil_image(byte_data):
    """Convierte contenido binario a imagen PIL"""
    return Image.open(BytesIO(byte_data)).convert("RGB")

# Convertir el contenido binario a imágenes PIL
df_pandas['image'] = df_pandas['content'].apply(bytes_to_pil_image)

# Convertir labels a integers según label2id
df_pandas['label_int'] = df_pandas['label'].map(label2id)

# Definir las features con ClassLabel para permitir estratificación
features = Features({
    'image': HFImage(),
    'label': ClassLabel(names=labels)
})

# Crear el dataset de Hugging Face con features explícitos
dataset = Dataset.from_dict(
    {
        'image': df_pandas['image'].tolist(),
        'label': df_pandas['label_int'].tolist()
    },
    features=features
)

print(f"\n✓ Dataset creado con {len(dataset)} imágenes")
print(f"Características: {dataset.features}")

# Dividir en train y validation (80-20) con estratificación
train_test_split = dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column='label')
train_dataset = train_test_split['train']
val_dataset = train_test_split['test']

print(f"\n✓ Dataset dividido:")
print(f"  Training: {len(train_dataset)} imágenes")
print(f"  Validation: {len(val_dataset)} imágenes")

Dataset cargado: 56 imágenes

Distribución por clase:
label
minor    21
major    18
ok       17
Name: count, dtype: int64

✓ Dataset creado con 56 imágenes
Características: {'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['ok', 'minor', 'major'])}

✓ Dataset dividido:
  Training: 44 imágenes
  Validation: 12 imágenes


In [0]:
# =====================================================
# Paso 8: Preprocesar imágenes SIN data augmentation
# =====================================================

def preprocess_images(examples):
    """
    Preprocesa imágenes sin augmentation.
    Solo redimensiona y normaliza según el processor del modelo.
    """
    processed = processor(examples['image'], return_tensors='pt')
    examples['pixel_values'] = [processed['pixel_values'][i] for i in range(len(examples['image']))]
    return examples

# Aplicar el mismo preprocesamiento a training y validation
print("Procesando training set (sin augmentation)...")
train_dataset_processed = train_dataset.map(
    preprocess_images, 
    batched=True, 
    batch_size=8,
    remove_columns=['image']
)

print("Procesando validation set (sin augmentation)...")
val_dataset_processed = val_dataset.map(
    preprocess_images, 
    batched=True, 
    batch_size=8,
    remove_columns=['image']
)

print("\n✓ Datasets preprocesados SIN augmentation")
print(f"  Training: {len(train_dataset_processed)} imágenes")
print(f"  Validation: {len(val_dataset_processed)} imágenes")
print(f"  Transformaciones: solo resize + normalización estándar")

Procesando training set (sin augmentation)...


Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Procesando validation set (sin augmentation)...


Map:   0%|          | 0/12 [00:00<?, ? examples/s]


✓ Datasets preprocesados SIN augmentation
  Training: 44 imágenes
  Validation: 12 imágenes
  Transformaciones: solo resize + normalización estándar


In [0]:
# =====================================================
# Paso 9: Configurar entrenamiento - EARLY STOPPING
# =====================================================

from transformers import DefaultDataCollator, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import os

# Desactivar entrenamiento distribuido
os.environ["RANK"] = "-1"
os.environ["WORLD_SIZE"] = "1"
os.environ["LOCAL_RANK"] = "-1"

data_collator = DefaultDataCollator()

def compute_metrics(eval_pred):
    """
    Calcula múltiples métricas para evaluación completa.
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Configuración con EARLY STOPPING
training_args = TrainingArguments(
    output_dir="./vehicle_damage_classifier_early_stop",
    num_train_epochs=20,               # Máximo 20 épocas
    per_device_train_batch_size=8,     
    per_device_eval_batch_size=8,     
    learning_rate=5e-5,                # Mismo que baseline exitoso (66.67%)
    weight_decay=0.01,                 
    logging_steps=3,                   
    eval_strategy="epoch",             # Evaluar cada época
    save_strategy="epoch",             
    load_best_model_at_end=True,       # Cargar el mejor modelo al final
    metric_for_best_model="accuracy",  # Métrica para determinar el mejor modelo
    greater_is_better=True,            
    save_total_limit=3,                
    remove_unused_columns=False,
    push_to_hub=False,
    dataloader_num_workers=0,
    fp16=False,
)

# Callback de Early Stopping
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=3,         # Detener si no mejora en 3 épocas
    early_stopping_threshold=0.0       # Cualquier mejora cuenta
)

print("✓ Parámetros de entrenamiento con EARLY STOPPING configurados")
print(f"  Épocas máximas: {training_args.num_train_epochs}")
print(f"  Early stopping patience: 3 épocas")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate} (baseline exitoso)")
print(f"  Weight decay: {training_args.weight_decay}")
print(f"  Augmentation: NINGUNO")
print(f"  Fine-tuning: 100% del modelo (sin freezing)")
print(f"\n⭐ Estrategia: Baseline + Early Stopping para prevenir overfitting")

✓ Parámetros de entrenamiento con EARLY STOPPING configurados
  Épocas máximas: 20
  Early stopping patience: 3 épocas
  Batch size: 8
  Learning rate: 5e-05 (baseline exitoso)
  Weight decay: 0.01
  Augmentation: NINGUNO
  Fine-tuning: 100% del modelo (sin freezing)

⭐ Estrategia: Baseline + Early Stopping para prevenir overfitting


In [0]:
# =====================================================
# Paso 10: Abrir corrida en MLflow y entrenar
# =====================================================

# Iniciar corrida de MLflow
with mlflow.start_run(run_name="vehicle_damage_resnet50_early_stop") as run:
    
    # Registrar parámetros del experimento
    mlflow.log_param("model_base", checkpoint)
    mlflow.log_param("num_labels", len(labels))
    mlflow.log_param("labels", str(labels))
    mlflow.log_param("max_epochs", training_args.num_train_epochs)
    mlflow.log_param("early_stopping_patience", 3)
    mlflow.log_param("batch_size", training_args.per_device_train_batch_size)
    mlflow.log_param("learning_rate", training_args.learning_rate)
    mlflow.log_param("train_size", len(train_dataset_processed))
    mlflow.log_param("val_size", len(val_dataset_processed))
    
    # Crear el Trainer con early stopping callback
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset_processed,
        eval_dataset=val_dataset_processed,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[early_stopping_callback]  # Agregar early stopping
    )
    
    print("✓ Trainer creado con early stopping")
    print("\nIniciando entrenamiento...")
    print("El entrenamiento se detendrá automáticamente si no mejora en 3 épocas\n")
    
    # Entrenar el modelo
    train_results = trainer.train()
    
    print("\n✓ Entrenamiento completado")
    
    # Obtener las métricas de evaluación del log history
    eval_results = {}
    for log in trainer.state.log_history:
        if 'eval_loss' in log:
            eval_results = {k: v for k, v in log.items() if k.startswith('eval_')}
    
    # Determinar cuántas épocas se entrenaron realmente
    epochs_trained = int(train_results.global_step / len(train_dataset_processed) * training_args.per_device_train_batch_size)
    
    print(f"\n✓ Épocas entrenadas: {epochs_trained} (de {training_args.num_train_epochs} máximo)")
    print("\n✓ Resultados de validación (mejor época):")
    for key, value in eval_results.items():
        print(f"  {key}: {value:.4f}")
        mlflow.log_metric(key, value)
    
    mlflow.log_metric("epochs_trained", epochs_trained)
    
    # Guardar el mejor modelo Y el preprocessor
    trainer.save_model("./best_model")
    processor.save_pretrained("./best_model")
    
    run_id = run.info.run_id
    print(f"\n✓ Modelo y preprocessor guardados en ./best_model")
    print(f"✓ Run ID: {run_id}")

✓ Trainer creado con early stopping

Iniciando entrenamiento...
El entrenamiento se detendrá automáticamente si no mejora en 3 épocas



/local_disk0/.ephemeral_nfs/envs/pythonEnv-9f28dda4-d5a0-420d-9316-4aa4e6bbfac0/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.102933,1.084480,0.583333,0.546032,0.722222,0.583333
2,1.086270,1.072590,0.583333,0.464646,0.390476,0.583333
3,1.074693,1.069079,0.583333,0.464646,0.390476,0.583333
4,1.086407,1.066092,0.750000,0.738889,0.805556,0.750000
5,1.056277,1.056471,0.750000,0.738889,0.805556,0.750000
6,1.063104,1.054108,0.750000,0.738889,0.805556,0.750000
7,1.071336,1.055410,0.750000,0.738889,0.805556,0.750000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/local_disk0/.ephemeral_nfs/envs/pythonEnv-9f28dda4-d5a0-420d-9316-4aa4e6bbfac0/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

/local_disk0/.ephemeral_nfs/envs/pythonEnv-9f28dda4-d5a0-420d-9316-4aa4e6bbfac0/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Entrenamiento completado

✓ Épocas entrenadas: 7 (de 20 máximo)

✓ Resultados de validación (mejor época):
  eval_loss: 1.0554
  eval_accuracy: 0.7500
  eval_f1: 0.7389
  eval_precision: 0.8056
  eval_recall: 0.7500
  eval_runtime: 2.7053
  eval_samples_per_second: 4.4360
  eval_steps_per_second: 0.7390


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Modelo y preprocessor guardados en ./best_model
✓ Run ID: f709fda03ee14ff4a154a8f12acea1f8


In [0]:
# =====================================================
# Paso 10.5: Restaurar modelo desde MLflow (75% accuracy)
# =====================================================
# El cluster se terminó y ./best_model se borró (almacenamiento efímero)
# Necesitamos descargar el modelo desde MLflow antes de continuar

import mlflow
import os
import shutil

print("Verificando si el modelo ya existe...")

if os.path.exists("./best_model"):
    print("✓ ./best_model ya existe (no es necesario descargar)\n")
else:
    print("✗ ./best_model no existe. Descargando desde MLflow...\n")
    
    # Run ID del modelo con 75% accuracy
    best_run_id = "f709fda03ee14ff4a154a8f12acea1f8"
    
    # Obtener información del run
    mlflow.set_experiment("/Shared/insurce_claims_experiment")
    run = mlflow.get_run(best_run_id)
    
    print(f"Descargando modelo del Run ID: {best_run_id}")
    print(f"  Accuracy: {run.data.metrics.get('eval_accuracy', 'N/A'):.2%}")
    print(f"  F1: {run.data.metrics.get('eval_f1', 'N/A'):.4f}")
    print(f"  Fecha: {run.info.start_time}\n")
    
    # El modelo fue guardado con trainer.save_model() en el output_dir
    # Los checkpoints están en los artifacts del run
    artifact_path = run.info.artifact_uri
    
    # Buscar el checkpoint en los artifacts
    # Durante el entrenamiento con early stopping, Trainer guarda checkpoints
    client = mlflow.tracking.MlflowClient()
    artifacts = client.list_artifacts(best_run_id)
    
    # Buscar el directorio del último checkpoint
    checkpoint_dirs = [a.path for a in artifacts if a.path.startswith('checkpoint-')]
    
    if checkpoint_dirs:
        # Usar el último checkpoint (mayor número)
        checkpoint_path = sorted(checkpoint_dirs)[-1]
        print(f"Descargando checkpoint: {checkpoint_path}...")
        
        # Descargar el checkpoint
        local_path = mlflow.artifacts.download_artifacts(
            artifact_uri=f"{artifact_path}/{checkpoint_path}",
            dst_path="."
        )
        
        # Renombrar a ./best_model
        if os.path.exists("./best_model"):
            shutil.rmtree("./best_model")
        shutil.move(local_path, "./best_model")
        
        print(f"✓ Modelo restaurado en ./best_model")
    else:
        print("⚠️ No se encontraron checkpoints en los artifacts.")
        print("   Intentando buscar en el output_dir del training_args...\n")
        
        # Intentar con el output_dir que se usó durante el entrenamiento
        # El modelo podría estar en dbfs:/databricks/mlflow-tracking/...
        print("El modelo se guardó localmente durante el entrenamiento.")
        print("Como el cluster se terminó, necesitamos re-entrenar.\n")
        print("OPCIÓN: Saltar a la celda 5 y re-ejecutar celdas 5-10 (~1 hora)")
        raise Exception("Modelo no disponible. Ver mensaje arriba para opciones.")

# Verificar archivos restaurados
if os.path.exists("./best_model"):
    files = os.listdir("./best_model")
    print(f"\n✓ Archivos en ./best_model: {len(files)}")
    for f in sorted(files):
        print(f"  - {f}")
    print("\n✓ Listo para continuar entrenamiento (+3 épocas)")

Verificando si el modelo ya existe...
✓ ./best_model ya existe (no es necesario descargar)


✓ Archivos en ./best_model: 4
  - config.json
  - model.safetensors
  - preprocessor_config.json
  - training_args.bin

✓ Listo para continuar entrenamiento (+3 épocas)


In [0]:
# =====================================================
# Continuar entrenamiento desde checkpoint: +3 épocas
# =====================================================
# Modelo actual: 75% accuracy (9/12 correctos)
# Objetivo: 90% accuracy (11/12 correctos) = +2 predicciones correctas

from transformers import AutoModelForImageClassification, AutoImageProcessor, Trainer, TrainingArguments
import mlflow
import os

# Desactivar entrenamiento distribuido
os.environ["RANK"] = "-1"
os.environ["WORLD_SIZE"] = "1"
os.environ["LOCAL_RANK"] = "-1"

print("Cargando modelo desde checkpoint...")

# Cargar el modelo guardado (75% accuracy)
model_continued = AutoModelForImageClassification.from_pretrained("./best_model")
processor_continued = AutoImageProcessor.from_pretrained("./best_model")

print("✓ Modelo y processor cargados desde ./best_model")
print(f"  Accuracy actual: 75% (9/12 correctos)")
print(f"  Objetivo: 90% (11/12 correctos)\n")

# Configurar 3 épocas adicionales
training_args_continued = TrainingArguments(
    output_dir="./vehicle_damage_classifier_continued",
    num_train_epochs=3,                # Solo 3 épocas más
    per_device_train_batch_size=8,     
    per_device_eval_batch_size=8,     
    learning_rate=3e-5,                # Learning rate más bajo para fine-tuning fino
    weight_decay=0.01,                 
    logging_steps=2,                   
    eval_strategy="epoch",             
    save_strategy="epoch",             
    load_best_model_at_end=True,      
    metric_for_best_model="accuracy", 
    greater_is_better=True,            
    save_total_limit=2,                
    remove_unused_columns=False,
    push_to_hub=False,
    dataloader_num_workers=0,
    fp16=False,
)

print("✓ Configuración para entrenamiento continuado:")
print(f"  Épocas adicionales: 3")
print(f"  Learning rate: 3e-5 (más conservador para ajuste fino)")
print(f"  Partiendo de accuracy: 75%\n")

# Entrenar con MLflow
with mlflow.start_run(run_name="vehicle_damage_resnet50_continued") as run:
    
    # Registrar parámetros
    mlflow.log_param("model_base", "./best_model (75% accuracy)")
    mlflow.log_param("num_labels", 3)
    mlflow.log_param("epochs", 3)
    mlflow.log_param("learning_rate", training_args_continued.learning_rate)
    mlflow.log_param("starting_accuracy", 0.75)
    mlflow.log_param("train_size", len(train_dataset_processed))
    mlflow.log_param("val_size", len(val_dataset_processed))
    
    # Crear trainer
    trainer_continued = Trainer(
        model=model_continued,
        args=training_args_continued,
        train_dataset=train_dataset_processed,
        eval_dataset=val_dataset_processed,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    
    print("Iniciando entrenamiento continuado...\n")
    
    # Entrenar
    train_results_continued = trainer_continued.train()
    
    print("\n✓ Entrenamiento continuado completado")
    
    # Obtener métricas finales
    eval_results_continued = {}
    for log in trainer_continued.state.log_history:
        if 'eval_loss' in log:
            eval_results_continued = {k: v for k, v in log.items() if k.startswith('eval_')}
    
    final_accuracy = eval_results_continued.get('eval_accuracy', 0)
    correct_predictions = int(final_accuracy * 12)
    
    print(f"\n✓ Resultados finales:")
    print(f"  Accuracy: {final_accuracy:.2%} ({correct_predictions}/12 correctos)")
    for key, value in eval_results_continued.items():
        if key != 'eval_accuracy':
            print(f"  {key}: {value:.4f}")
        mlflow.log_metric(key, value)
    
    # Guardar si mejoró
    if final_accuracy >= 0.75:
        trainer_continued.save_model("./best_model")
        processor_continued.save_pretrained("./best_model")
        print(f"\n✓ Modelo mejorado guardado en ./best_model")
        
        if final_accuracy >= 0.90:
            print("\n🎯 ¡OBJETIVO ALCANZADO! Accuracy >= 90%")
        elif final_accuracy > 0.75:
            print(f"\n📈 Mejora lograda: 75% → {final_accuracy:.2%}")
        else:
            print(f"\n📊 Accuracy mantenido en 75%")
    else:
        print(f"\n⚠️ Accuracy bajó a {final_accuracy:.2%}. No se actualiza el modelo.")
    
    run_id_continued = run.info.run_id
    print(f"\nRun ID: {run_id_continued}")

Cargando modelo desde checkpoint...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

✓ Modelo y processor cargados desde ./best_model
  Accuracy actual: 75% (9/12 correctos)
  Objetivo: 90% (11/12 correctos)

✓ Configuración para entrenamiento continuado:
  Épocas adicionales: 3
  Learning rate: 3e-5 (más conservador para ajuste fino)
  Partiendo de accuracy: 75%



In [0]:
# =====================================================
# Paso 11: Construir pipeline de inferencia
# =====================================================

from transformers import pipeline

# Crear pipeline de clasificación de imágenes
inference_pipeline = pipeline(
    "image-classification",
    model="./best_model",
    device=-1  # CPU, cambiar a 0 para GPU
)

print("✓ Pipeline de inferencia creado")

# =====================================================
# Paso 12: Envolver el modelo para MLflow
# =====================================================

class VehicleDamageClassifierWrapper(mlflow.pyfunc.PythonModel):
    """
    Wrapper del modelo para MLflow.
    Recibe imágenes binarias y devuelve predicciones.
    """
    
    def load_context(self, context):
        """Carga el pipeline cuando se carga el modelo"""
        from transformers import pipeline
        self.pipeline = pipeline(
            "image-classification",
            model=context.artifacts["model_path"],
            device=-1
        )
    
    def predict(self, context, model_input):
        """
        Predice la clase de daño para cada imagen.
        
        Parámetros:
        - model_input: pandas DataFrame con columna 'content' (bytes de imagen)
        
        Retorna:
        - pandas DataFrame con columnas: label, score
        """
        from PIL import Image
        from io import BytesIO
        import pandas as pd
        
        results = []
        
        for _, row in model_input.iterrows():
            # Convertir bytes a imagen PIL
            image = Image.open(BytesIO(row['content'])).convert("RGB")
            
            # Predecir
            predictions = self.pipeline(image)
            
            # Tomar la predicción con mayor score
            top_prediction = predictions[0]
            
            results.append({
                'label': top_prediction['label'],
                'score': top_prediction['score']
            })
        
        return pd.DataFrame(results)

print("✓ Wrapper del modelo creado")

# =====================================================
# Paso 13: Definir muestra de entrada y firma
# =====================================================

# Tomar una muestra de imágenes para la firma
sample_images = df_pandas[['content']].head(3)

# Hacer una predicción de prueba con el wrapper
wrapped_model = VehicleDamageClassifierWrapper()
wrapped_model.pipeline = inference_pipeline
sample_predictions = wrapped_model.predict(None, sample_images)

print("✓ Muestra de entrada y predicciones de prueba:")
print(sample_predictions)

# Inferir la firma del modelo
signature = infer_signature(sample_images, sample_predictions)

print("\n✓ Firma del modelo inferida")
print(f"  Inputs: {signature.inputs}")
print(f"  Outputs: {signature.outputs}")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

✓ Pipeline de inferencia creado
✓ Wrapper del modelo creado


/local_disk0/.ephemeral_nfs/envs/pythonEnv-ba896941-95bd-4063-8ef9-36cfe2e00bc7/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


✓ Muestra de entrada y predicciones de prueba:
   label     score
0  minor  0.338416
1  major  0.342544
2  major  0.360639

✓ Firma del modelo inferida
  Inputs: ['content': binary (required)]
  Outputs: ['label': string (required), 'score': double (required)]


In [0]:
# =====================================================
# Paso 14: Registrar el modelo en MLflow
# =====================================================

import mlflow.pyfunc

with mlflow.start_run(run_name="model_registration") as run:
    
    # Registrar el modelo como PyFunc
    mlflow.pyfunc.log_model(
        artifact_path="vehicle_damage_classifier",
        python_model=VehicleDamageClassifierWrapper(),
        artifacts={
            "model_path": "./best_model"
        },
        signature=signature,
        input_example=sample_images,
        pip_requirements=[
            "transformers",
            "torch",
            "pillow",
            "pandas"
        ]
    )
    
    model_uri = f"runs:/{run.info.run_id}/vehicle_damage_classifier"
    print(f"✓ Modelo registrado en MLflow")
    print(f"  Model URI: {model_uri}")
    
    registration_run_id = run.info.run_id

print("\n" + "="*70)

# =====================================================
# Paso 15: Publicar el modelo en Unity Catalog
# =====================================================

from mlflow import MlflowClient

client = MlflowClient()

# Nombre del modelo en Unity Catalog
model_name = "smart_claims.gold.vehicle_damage_classifier"

# Registrar el modelo en Unity Catalog
model_version = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print(f"✓ Modelo registrado en Unity Catalog")
print(f"  Nombre: {model_name}")
print(f"  Versión: {model_version.version}")

# Asignar alias 'prod' a esta versión
client.set_registered_model_alias(
    name=model_name,
    alias="prod",
    version=model_version.version
)

print(f"\n✓ Alias 'prod' asignado a versión {model_version.version}")
print(f"\nPara usar el modelo:")
print(f"  model = mlflow.pyfunc.load_model('models:/{model_name}@prod')")

2026/04/23 20:40:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-45b33c63-ec1b.cloud.databricks.com/ml/experiments/2741300458728716/models/m-4e2a03d3d1114702b2b7c756df156223?o=7474659143590139
2026/04/23 20:40:59 INFO mlflow.pyfunc: Validating input example against model signature


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

✓ Modelo registrado en MLflow
  Model URI: runs:/8d3880ad90254a6fb27766506a718c9e/vehicle_damage_classifier



Successfully registered model 'smart_claims.gold.vehicle_damage_classifier'.
2026/04/23 20:41:08 WARNING mlflow.tracking._model_registry.fluent: Run with id 8d3880ad90254a6fb27766506a718c9e has no artifacts at artifact path 'vehicle_damage_classifier', registering model based on models:/m-4e2a03d3d1114702b2b7c756df156223 instead


Uploading artifacts:   0%|          | 0/16 [00:00<?, ?it/s]

🔗 Created version '1' of model 'smart_claims.gold.vehicle_damage_classifier': https://dbc-45b33c63-ec1b.cloud.databricks.com/explore/data/models/smart_claims/gold/vehicle_damage_classifier/version/1?o=7474659143590139


✓ Modelo registrado en Unity Catalog
  Nombre: smart_claims.gold.vehicle_damage_classifier
  Versión: 1

✓ Alias 'prod' asignado a versión 1

Para usar el modelo:
  model = mlflow.pyfunc.load_model('models:/smart_claims.gold.vehicle_damage_classifier@prod')


In [0]:
# =====================================================
# Paso 16: Crear la función de predicción en Spark
# =====================================================

import mlflow.pyfunc
from pyspark.sql.functions import pandas_udf, col, struct
from pyspark.sql.types import StructType, StructField, StringType, FloatType
import pandas as pd

# Definir el nombre del modelo en Unity Catalog
model_name = "smart_claims.gold.vehicle_damage_classifier"

# Cargar el modelo desde Unity Catalog usando el alias prod
loaded_model = mlflow.pyfunc.load_model(f"models:/{model_name}@prod")

print(f"✓ Modelo cargado desde Unity Catalog")
print(f"  Modelo: {model_name}@prod")

# Definir el esquema de salida de la UDF
result_schema = StructType([
    StructField("predicted_label", StringType(), True),
    StructField("confidence_score", FloatType(), True)
])

# Crear la UDF de predicción
@pandas_udf(result_schema)
def predict_damage_udf(content_series: pd.Series) -> pd.DataFrame:
    """
    UDF de Spark que predice el nivel de daño vehicular.
    
    Parámetros:
    - content_series: Serie de Pandas con contenido binario de imágenes
    
    Retorna:
    - DataFrame de Pandas con predicted_label y confidence_score
    """
    # Preparar input en el formato esperado por el modelo
    input_df = pd.DataFrame({'content': content_series})
    
    # Predecir
    predictions = loaded_model.predict(input_df)
    
    # Renombrar columnas para que coincidan con el esquema
    predictions = predictions.rename(columns={
        'label': 'predicted_label',
        'score': 'confidence_score'
    })
    
    return predictions

print("✓ Función de predicción UDF creada")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

✓ Modelo cargado desde Unity Catalog
  Modelo: smart_claims.gold.vehicle_damage_classifier@prod
✓ Función de predicción UDF creada


In [0]:
# =====================================================
# Paso 17: Predecir sobre imágenes reales de reclamos
# =====================================================

# Leer la tabla de claim_images desde Silver
df_claim_images = spark.table("smart_claims.silver.claim_images")

print(f"Imágenes de reclamos cargadas: {df_claim_images.count()}")

# Aplicar el modelo a las imágenes
df_predictions = df_claim_images.withColumn(
    "prediction",
    predict_damage_udf(col("content"))
).select(
    "image_name",
    "path",
    col("prediction.predicted_label").alias("predicted_damage_level"),
    col("prediction.confidence_score").alias("prediction_confidence")
)

print("✓ Predicciones generadas")
print(f"\nPrimeras predicciones:")
display(df_predictions.limit(10))

Imágenes de reclamos cargadas: 15
✓ Predicciones generadas

Primeras predicciones:


image_name,path,predicted_damage_level,prediction_confidence
1_High.jpg,dbfs:/Volumes/smart_claims/landing/landing/claim_images/1_High.jpg,minor,0.35383958
1_Low.jpg,dbfs:/Volumes/smart_claims/landing/landing/claim_images/1_Low.jpg,ok,0.3537034
1_Medium.jpg,dbfs:/Volumes/smart_claims/landing/landing/claim_images/1_Medium.jpg,minor,0.34220147
2_High.jpg,dbfs:/Volumes/smart_claims/landing/landing/claim_images/2_High.jpg,ok,0.342183
2_Low.jpg,dbfs:/Volumes/smart_claims/landing/landing/claim_images/2_Low.jpg,major,0.35466462
2_Medium.jpg,dbfs:/Volumes/smart_claims/landing/landing/claim_images/2_Medium.jpg,major,0.35070324
3_High.jpg,dbfs:/Volumes/smart_claims/landing/landing/claim_images/3_High.jpg,major,0.3417953
3_Low.jpg,dbfs:/Volumes/smart_claims/landing/landing/claim_images/3_Low.jpg,minor,0.3428786
3_Medium.jpg,dbfs:/Volumes/smart_claims/landing/landing/claim_images/3_Medium.jpg,ok,0.35070387
4_High.jpg,dbfs:/Volumes/smart_claims/landing/landing/claim_images/4_High.jpg,major,0.34010246


In [0]:
# =====================================================
# Paso 18: Unir la predicción con la metadata de imágenes
# =====================================================

from pyspark.sql.functions import lower

# Leer la tabla de metadata de imágenes desde Silver
df_metadata = spark.table("smart_claims.silver.claim_images_metadata_clean")

print(f"Metadata de imágenes cargada: {df_metadata.count()} registros")

# SOLUCIÓN: Normalizar ambos image_name a minúsculas antes del join
# Problema: claim_images tiene "4_High.jpg" pero metadata tiene "4_high.jpg"
df_predictions_normalized = df_predictions.withColumn(
    "image_name_lower",
    lower(col("image_name"))
)

df_metadata_normalized = df_metadata.withColumn(
    "image_name_lower",
    lower(col("image_name"))
)

# Unir predicciones con metadata usando image_name normalizado
df_predictions_with_metadata = df_predictions_normalized.join(
    df_metadata_normalized,
    on="image_name_lower",
    how="inner"
).select(
    df_predictions_normalized["image_name"],  # Mantener nombre original
    "claim_no",
    "chassis_no",
    "predicted_damage_level",
    "prediction_confidence",
    df_predictions_normalized["path"]
)

print("✓ Predicciones unidas con metadata (nombres normalizados a minúsculas)")
print(f"  Total registros: {df_predictions_with_metadata.count()}")
print(f"\nMuestra de predicciones con metadata:")
display(df_predictions_with_metadata.limit(10))

# Verificar distribución de predicciones
print("\nDistribución de predicciones:")
display(
    df_predictions_with_metadata
    .groupBy("predicted_damage_level")
    .count()
    .orderBy("predicted_damage_level")
)

Metadata de imágenes cargada: 13001 registros
✓ Predicciones unidas con metadata (nombres normalizados a minúsculas)
  Total registros: 13001

Muestra de predicciones con metadata:


image_name,claim_no,chassis_no,predicted_damage_level,prediction_confidence,path
1_High.jpg,891ca2c0-68a2-423c-a8c0-e69fa34f522d,ME4KC20C7KA003646,minor,0.35383958,dbfs:/Volumes/smart_claims/landing/landing/claim_images/1_High.jpg
1_Low.jpg,172c0cc8-d9b9-4930-8870-929547dd86bd,LSJA16E36KG077424,ok,0.3537034,dbfs:/Volumes/smart_claims/landing/landing/claim_images/1_Low.jpg
1_Medium.jpg,f2626bbf-b54d-476f-bb0d-33126abcb4cb,WBAKR0106J0X59330,minor,0.34220147,dbfs:/Volumes/smart_claims/landing/landing/claim_images/1_Medium.jpg
2_High.jpg,410f675c-e490-4bc3-8523-8acd83ebf9de,2C3CDZBT8KH545623,ok,0.342183,dbfs:/Volumes/smart_claims/landing/landing/claim_images/2_High.jpg
2_Low.jpg,139acbf7-5df4-4a2f-9560-63fe35c6604d,JTEBU11F9LK254235,major,0.35466462,dbfs:/Volumes/smart_claims/landing/landing/claim_images/2_Low.jpg
2_Medium.jpg,83157c34-9212-4946-8bc4-64b799e47dbb,LSJA24U60LS012452,major,0.35070324,dbfs:/Volumes/smart_claims/landing/landing/claim_images/2_Medium.jpg
3_High.jpg,43221a12-7df3-4ae2-bdc3-bda783a68070,1C4HJXDG7JW208374,major,0.3417953,dbfs:/Volumes/smart_claims/landing/landing/claim_images/3_High.jpg
3_Low.jpg,4bbd752f-2f07-49a5-8fed-8bda73ab826c,WMWXM5106FT925238,minor,0.3428786,dbfs:/Volumes/smart_claims/landing/landing/claim_images/3_Low.jpg
3_Medium.jpg,d56437f5-c21b-48bf-b5e9-94976b9090d9,ZAREAEAN2K7613358,ok,0.35070387,dbfs:/Volumes/smart_claims/landing/landing/claim_images/3_Medium.jpg
4_High.jpg,173d04d3-592c-4a12-af88-bc8422bfd8e9,LSJA24W3XKS013237,major,0.34010246,dbfs:/Volumes/smart_claims/landing/landing/claim_images/4_High.jpg



Distribución de predicciones:


predicted_damage_level,count
major,4457
minor,816
ok,7728


In [0]:
# =====================================================
# Paso 19: Guardar la tabla final de predicciones
# =====================================================

from pyspark.sql.functions import current_timestamp

# Agregar timestamp de predicción
df_final_predictions = df_predictions_with_metadata.withColumn(
    "prediction_timestamp",
    current_timestamp()
)

# Guardar en Gold
table_name = "smart_claims.gold.claim_damage_predictions"

df_final_predictions.write.mode("overwrite").saveAsTable(table_name)

print("✓ Tabla final de predicciones guardada")
print(f"  Tabla: {table_name}")
print(f"  Registros: {df_final_predictions.count()}")
print(f"\nColumnas finales:")
for col_name in df_final_predictions.columns:
    print(f"  - {col_name}")

print("\n" + "="*70)
print("✅ PROCESO COMPLETADO")
print("="*70)
print("\nLa tabla Gold con predicciones está lista para:")
print("  1. Aplicar reglas de negocio")
print("  2. Validación automática de reclamos")
print("  3. Detección de fraude")
print("  4. Dashboards y reportes")
print(f"\n➡️  Tabla: {table_name}")

✓ Tabla final de predicciones guardada
  Tabla: smart_claims.gold.claim_damage_predictions
  Registros: 13001

Columnas finales:
  - image_name
  - claim_no
  - chassis_no
  - predicted_damage_level
  - prediction_confidence
  - path
  - prediction_timestamp

✅ PROCESO COMPLETADO

La tabla Gold con predicciones está lista para:
  1. Aplicar reglas de negocio
  2. Validación automática de reclamos
  3. Detección de fraude
  4. Dashboards y reportes

➡️  Tabla: smart_claims.gold.claim_damage_predictions
